<a href="https://colab.research.google.com/github/franz-ops/IoT-Traffic-Analyzer/blob/main/Iot_23_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn import preprocessing

In [ ]:
!wget https://github.com/franz-ops/IoT-Traffic-Analyzer/raw/main/Iot-23-TS.zip

--2022-03-28 16:07:06--  https://github.com/franz-ops/IoT-Traffic-Analyzer/raw/main/Iot-23-TS.zip
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/franz-ops/IoT-Traffic-Analyzer/main/Iot-23-TS.zip [following]
--2022-03-28 16:07:06--  https://raw.githubusercontent.com/franz-ops/IoT-Traffic-Analyzer/main/Iot-23-TS.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26192940 (25M) [application/zip]
Saving to: ‘Iot-23-TS.zip’

Iot-23-TS.zip       100%[===================>]  24.98M  --.-KB/s    in 0.08s   

2022-03-28 16:07:06 (305 MB/s) - ‘Iot-23-TS.zip’ saved [26192940/26192940]



In [ ]:
import zipfile
zip_file = "Iot-23-TS.zip"
 
try:
    with zipfile.ZipFile(zip_file) as z:
        z.extractall()
        print("Extracted all")
except:
    print("Invalid file")

Extracted all


In [ ]:
df = pd.read_csv("Iot-23-TS.csv",low_memory=False)

In [ ]:
df = df.drop(axis=1, columns=["Unnamed: 0"])

In [ ]:
df

,id.resp_p,duration,orig_bytes,resp_bytes,missed_bytes,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,history_Ss,...,conn_state_SHR,service_dhcp,service_dns,service_http,service_irc,service_ssh,service_ssl,service_unknown,label,detailed-label
0,10,30.004642,8768.0,0.0,0,1,9216,0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,10,0.004564,0.0,3900.0,0,0,0,1,4264,0,...,1,1,0,0,0,0,0,0,0,0
2,7,3.948539,876.0,0.0,0,0,1164,0,0,0,...,0,0,1,0,0,0,0,0,0,0
3,7,3.768179,876.0,0.0,0,0,1164,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,7,0.000114,451.0,0.0,0,1,979,0,0,0,...,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1446656,6,0.002732,48.0,48.0,0,0,76,0,76,0,...,0,0,0,0,0,0,0,1,0,0
1446657,6,0.058227,48.0,48.0,0,0,76,0,76,0,...,0,0,0,0,0,0,0,1,0,0
1446658,6,0.002738,48.0,48.0,0,0,76,0,76,0,...,0,0,0,0,0,0,0,1,0,0
1446659,6,0.033229,48.0,48.0,0,0,76,0,76,0,...,0,0,0,0,0,0,0,1,0,0


In [ ]:
indexes = df[(df['detailed-label'] == 'C&C-FileDownload') | 
             (df['detailed-label'] == 'C&C-Torii') |
             (df['detailed-label'] == 'FileDownload') | 
             (df['detailed-label'] == 'C&C-HeartBeat-FileDownload') | 
             (df['detailed-label'] == 'C&C-Mirai') |
             (df['detailed-label'] == 'C&C-HeartBeat')].index
 
#droping mutiple rows based on column value
df.drop(indexes,inplace=True)

In [ ]:
df.isnull().sum()

ts                0
id.resp_p         0
proto             0
service           0
duration          0
orig_bytes        0
resp_bytes        0
conn_state        0
missed_bytes      0
history           0
orig_pkts         0
orig_ip_bytes     0
resp_pkts         0
resp_ip_bytes     0
label             0
detailed-label    0
dtype: int64

In [ ]:
df = df.drop(axis=1, columns=["tunnel_parents","id.orig_h","id.resp_h","local_orig","local_resp","id.orig_p"])
df['duration'] = df['duration'].fillna(pd.Timedelta(seconds=0))
df['orig_bytes'] = df['orig_bytes'].fillna(0)
df['resp_bytes'] = df['resp_bytes'].fillna(0)
df['resp_ip_bytes'] = df['resp_ip_bytes'].fillna(0)
df['orig_ip_bytes'] = df['orig_ip_bytes'].fillna(0) 
df['conn_state'] = df['conn_state'].fillna('-') 
df['history'] = df['history'].fillna('-')
df['service'] = df['service'].fillna("unknown")
# duration
df['duration'].replace(to_replace=['-'], value=0, inplace=True)
# orig_bytes
df['orig_bytes'].replace(to_replace=['-'], value=0, inplace=True)
# orig_bytes
df['resp_bytes'].replace(to_replace=['-'], value=0, inplace=True)

In [ ]:
orig_pkts_df = df.loc[:,('orig_pkts')].to_frame()
orig_pkts_df.rename(columns={'orig_pkts':'_orig_pkts_'}, inplace=True)

orig_pkts_df['orig_pkts']='0_10'
orig_pkts_df.loc[orig_pkts_df._orig_pkts_ > 10, 'orig_pkts'] = '10_10000'
orig_pkts_df.loc[orig_pkts_df._orig_pkts_ > 10000, 'orig_pkts'] = '>10000'

df['orig_pkts'] = orig_pkts_df['orig_pkts']
del orig_pkts_df

resp_pkts_df = df.loc[:,('resp_pkts')].to_frame()
resp_pkts_df.rename(columns={'resp_pkts':'_resp_pkts_'}, inplace=True)

resp_pkts_df['resp_pkts']='0_10'
resp_pkts_df.loc[resp_pkts_df._resp_pkts_ > 10, 'resp_pkts'] = '10_10000'
resp_pkts_df.loc[resp_pkts_df._resp_pkts_ > 10000, 'resp_pkts'] = '>10000'

df['resp_pkts'] = resp_pkts_df['resp_pkts']
del resp_pkts_df

In [ ]:
df

,ts,id.resp_p,proto,service,duration,orig_bytes,resp_bytes,conn_state,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,label,detailed-label
0,2019-07-03 13:16:59.172195072,67,udp,dhcp,0 days 00:00:30.004642,8768.0,0.0,S0,0,D,10_10000,9216,0_10,0,benign,-
1,2019-07-03 13:17:29.173340160,67,udp,dhcp,0 days 00:00:00.004564,0.0,3900.0,SHR,0,^d,0_10,0,10_10000,4264,benign,-
2,2019-07-03 13:19:13.959668992,5353,udp,dns,0 days 00:00:03.948539,876.0,0.0,S0,0,D,0_10,1164,0_10,0,benign,-
3,2019-07-03 13:19:58.302953984,5353,udp,dns,0 days 00:00:03.768179,876.0,0.0,S0,0,D,0_10,1164,0_10,0,benign,-
4,2019-07-03 13:20:24.472592128,5353,udp,dns,0 days 00:00:00.000114,451.0,0.0,S0,0,D,10_10000,979,0_10,0,benign,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1446656,2018-10-03 11:07:26.549982976,123,udp,NaN,0 days 00:00:00.002732,48.0,48.0,SF,0,Dd,0_10,76,0_10,76,Benign,-
1446657,2018-10-03 11:08:31.550179072,123,udp,NaN,0 days 00:00:00.058227,48.0,48.0,SF,0,Dd,0_10,76,0_10,76,Benign,-
1446658,2018-10-03 11:09:36.550151936,123,udp,NaN,0 days 00:00:00.002738,48.0,48.0,SF,0,Dd,0_10,76,0_10,76,Benign,-
1446659,2018-10-03 11:10:43.549967104,123,udp,NaN,0 days 00:00:00.033229,48.0,48.0,SF,0,Dd,0_10,76,0_10,76,Benign,-


In [ ]:
# Converto la durata da TimeDelta64 a secondi
df['duration'] = df[:]['duration'] / np.timedelta64(1, 's')

In [ ]:
id_resp_p_df = df.loc[:,('id.resp_p')].to_frame()
id_resp_p_df.rename(columns={'id.resp_p':'_id_resp_p_'}, inplace=True)
id_resp_p_df['id.resp_p']='gt_65535'
id_resp_p_df.loc[id_resp_p_df._id_resp_p_ < 65536, 'id.resp_p'] = 'registered_p'
id_resp_p_df.loc[id_resp_p_df._id_resp_p_ < 1024, 'id.resp_p'] = 'well_known_p'
    # ssh
id_resp_p_df.loc[id_resp_p_df._id_resp_p_ == 22, 'id.resp_p'] = 'ssh'

# telnet
id_resp_p_df.loc[id_resp_p_df._id_resp_p_ == 23, 'id.resp_p'] = 'telnet'

# Message Processing Module [recv]
id_resp_p_df.loc[id_resp_p_df._id_resp_p_ == 45, 'id.resp_p'] = 'mpm'

# Domain Name Server
id_resp_p_df.loc[id_resp_p_df._id_resp_p_ == 53, 'id.resp_p'] = 'dns'

# World Wide Web HTTP 
id_resp_p_df.loc[id_resp_p_df._id_resp_p_ == 80, 'id.resp_p'] = 'http_80'

# ntp
id_resp_p_df.loc[id_resp_p_df._id_resp_p_ == 123, 'id.resp_p'] = 'ntp'

  # https
id_resp_p_df.loc[id_resp_p_df._id_resp_p_ == 443, 'id.resp_p'] = 'https'

  # mdqs
id_resp_p_df.loc[id_resp_p_df._id_resp_p_ == 666, 'id.resp_p'] = 'mdqs'

# World Wide Web HTTP 
id_resp_p_df.loc[id_resp_p_df._id_resp_p_ == 8080, 'id.resp_p'] = 'http_8080'

# World Wide Web HTTP 
id_resp_p_df.loc[id_resp_p_df._id_resp_p_ == 8081, 'id.resp_p'] = 'http_8081'

df['id.resp_p'] = id_resp_p_df['id.resp_p']

In [ ]:
# history
#   a SYN w/o the ACK bit set

history_df = df.loc[:,('history')].to_frame()

history_df['history_Ss'] = history_df['history'].replace(regex='[Ss]', value=1)
history_df['history_Ss'].replace(regex='[^1]', value=0, inplace=True)

#   a SYN+ACK (“handshake”)
history_df['history_Hh'] = history_df['history'].replace(regex='[Hh]', value=1)
history_df['history_Hh'].replace(regex='[^1]', value=0, inplace=True)

#   a pure ACK
history_df['history_Aa'] = history_df['history'].replace(regex='[Aa]', value=1)
history_df['history_Aa'].replace(regex='[^1]', value=0, inplace=True)

#   packet with payload (“data”)
history_df['history_Dd'] = history_df['history'].replace(regex='[Dd]', value=1)
history_df['history_Dd'].replace(regex='[^1]', value=0, inplace=True)

#   packet with FIN bit set
history_df['history_Ff'] = history_df['history'].replace(regex='[Ff]', value=1)
history_df['history_Ff'].replace(regex='[^1]', value=0, inplace=True)

#   packet with RST bit set
history_df['history_Rr'] = history_df['history'].replace(regex='[Rr]', value=1)
history_df['history_Rr'].replace(regex='[^1]', value=0, inplace=True)

#   packet with a bad checksum (applies to UDP too)
history_df['history_Cc'] = history_df['history'].replace(regex='[Cc]', value=1)
history_df['history_Cc'].replace(regex='[^1]', value=0, inplace=True)

#   a content gap
history_df['history_Gg'] = history_df['history'].replace(regex='[Gg]', value=1)
history_df['history_Gg'].replace(regex='[^1]', value=0, inplace=True)

#   packet with retransmitted payload
history_df['history_Tt'] = history_df['history'].replace(regex='[Tt]', value=1)
history_df['history_Tt'].replace(regex='[^1]', value=0, inplace=True)

#   packet with a zero window advertisement
history_df['history_Ww'] = history_df['history'].replace(regex='[Ww]', value=1)
history_df['history_Ww'].replace(regex='[^1]', value=0, inplace=True)

#   inconsistent packet (e.g. FIN+RST bits set)
history_df['history_Ii'] = history_df['history'].replace(regex='[Ii]', value=1)
history_df['history_Ii'].replace(regex='[^1]', value=0, inplace=True)

#   multi-flag packet (SYN+FIN or SYN+RST bits set)
history_df['history_Qq'] = history_df['history'].replace(regex='[Qq]', value=1)
history_df['history_Qq'].replace(regex='[^1]', value=0, inplace=True)

#   connection direction was flipped by Zeek’s heuristic
history_df['history_conn_flip'] = history_df['history'].replace(regex='[.^]', value=1)
history_df['history_conn_flip'].replace(regex='[^1]', value=0, inplace=True)

#   - dash -
history_df['history_unknown'] = history_df['history'].replace(to_replace=['-'], value=1)
history_df['history_unknown'].replace(regex='[^1]', value=0, inplace=True)

history_df.drop(['history'], axis=1, inplace=True)

In [ ]:
df = df.join(history_df)
df = df.drop(axis=1, columns=["history"])

In [ ]:
df = pd.get_dummies(df, columns=['proto'])
df = pd.get_dummies(df, columns=['conn_state'])
df = pd.get_dummies(df, columns=['service'])

In [ ]:
label = df.pop("label")
attack = df.pop("detailed-label")
df = df.join(label)
df = df.join(attack)
df

,id.resp_p,duration,orig_bytes,resp_bytes,missed_bytes,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,history_Ss,...,conn_state_SHR,service_dhcp,service_dns,service_http,service_irc,service_ssh,service_ssl,service_unknown,label,detailed-label
0,well_known_p,30.004642,8768.0,0.0,0,10_10000,9216,0_10,0,0,...,0,1,0,0,0,0,0,0,benign,-
1,well_known_p,0.004564,0.0,3900.0,0,0_10,0,10_10000,4264,0,...,1,1,0,0,0,0,0,0,benign,-
2,registered_p,3.948539,876.0,0.0,0,0_10,1164,0_10,0,0,...,0,0,1,0,0,0,0,0,benign,-
3,registered_p,3.768179,876.0,0.0,0,0_10,1164,0_10,0,0,...,0,0,1,0,0,0,0,0,benign,-
4,registered_p,0.000114,451.0,0.0,0,10_10000,979,0_10,0,0,...,0,0,1,0,0,0,0,0,benign,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1446656,ntp,0.002732,48.0,48.0,0,0_10,76,0_10,76,0,...,0,0,0,0,0,0,0,1,Benign,-
1446657,ntp,0.058227,48.0,48.0,0,0_10,76,0_10,76,0,...,0,0,0,0,0,0,0,1,Benign,-
1446658,ntp,0.002738,48.0,48.0,0,0_10,76,0_10,76,0,...,0,0,0,0,0,0,0,1,Benign,-
1446659,ntp,0.033229,48.0,48.0,0,0_10,76,0_10,76,0,...,0,0,0,0,0,0,0,1,Benign,-


In [ ]:
def get_integer_mapping(le):
    '''
    Return a dict mapping labels to their integer values
    from an SKlearn LabelEncoder
    le = a fitted SKlearn LabelEncoder
    '''
    res = {}
    for cl in le.classes_:
        res.update({cl:le.transform([cl])[0]})

    return res

In [ ]:
# Creiamo una One Hot Encoding per la colonna Label
df['label'] = df['label'].str.title()
#dummy1 = pd.get_dummies(df['label'], drop_first=True)
#df['label'] = dummy1
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
print("Label Mapping:")
print(get_integer_mapping(le))

# Creiamo una One Hot Encoding per la colonna detailed-label e proto
le = LabelEncoder()
df['detailed-label'] = le.fit_transform(df['detailed-label'])
print("Detailed-Label Mapping:")
get_integer_mapping(le)

Label Mapping:
{'Benign': 0, 'Malicious': 1}
Detailed-Label Mapping:


{'-': 0,
 'Attack': 1,
 'C&C': 2,
 'DDoS': 3,
 'Okiru': 4,
 'PartOfAHorizontalPortScan': 5}

In [ ]:
df[['detailed-label']].value_counts(dropna=False)

detailed-label
5                 825931
4                 262687
0                 199790
3                 138775
2                  15107
1                   3915
dtype: int64

In [ ]:
#df['proto'] = LabelEncoder().fit_transform(df['proto'])
#df['service'] = LabelEncoder().fit_transform(df['service'])
#df['conn_state'] = LabelEncoder().fit_transform(df['conn_state'])
df['orig_pkts'] = LabelEncoder().fit_transform(df['orig_pkts'])
df['id.resp_p'] = LabelEncoder().fit_transform(df['id.resp_p'])
#df['id.orig_p'] = LabelEncoder().fit_transform(df['id.orig_p'])
df['resp_pkts'] = LabelEncoder().fit_transform(df['resp_pkts'])

In [ ]:
df = df.drop(axis=1, columns=["ts"])

In [ ]:
#scalar = ["id.resp_p","duration","orig_bytes","resp_bytes","missed_bytes","orig_pkts","orig_ip_bytes","resp_pkts","resp_ip_bytes"]

In [ ]:
'''# PREPROCESSING
X = df[scalar]
scaler = preprocessing.Normalizer()
scaler.fit(X)
X_scaled = scaler.transform(X)
df[scalar] = X_scaled'''

In [ ]:
# PREPROCESSING
X = df[df.columns[:-2]]
scaler = preprocessing.Normalizer()
scaler.fit(X)
X_scaled = scaler.transform(X)
df[df.columns[:-2]] = X_scaled

In [ ]:
df.to_csv("Iot-23-prova.csv")

In [ ]:
!zip Iot-23-prova.zip Iot-23-prova.csv

  adding: Iot-23-prova.csv (deflated 97%)


In [ ]:
df

,id.resp_p,duration,orig_bytes,resp_bytes,missed_bytes,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,history_Ss,...,conn_state_SHR,service_dhcp,service_dns,service_http,service_irc,service_ssh,service_ssl,service_unknown,label,detailed-label
0,0.000786,2.358746e-03,0.689276,0.000000,0.0,0.000079,0.724495,0.000000,0.000000,0.0,...,0.000000,0.000079,0.000000,0.0,0.0,0.0,0.0,0.000000,0,0
1,0.001731,7.898156e-07,0.000000,0.674908,0.0,0.000000,0.000000,0.000173,0.737900,0.0,...,0.000173,0.000173,0.000000,0.0,0.0,0.0,0.0,0.000000,0,0
2,0.004805,2.710372e-03,0.601307,0.000000,0.0,0.000000,0.798998,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000686,0.0,0.0,0.0,0.0,0.000000,0,0
3,0.004805,2.586569e-03,0.601308,0.000000,0.0,0.000000,0.798998,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000686,0.0,0.0,0.0,0.0,0.000000,0,0
4,0.006494,1.057599e-07,0.418401,0.000000,0.0,0.000928,0.908237,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000928,0.0,0.0,0.0,0.0,0.000000,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1446656,0.047140,2.146462e-05,0.377124,0.377124,0.0,0.000000,0.597112,0.000000,0.597112,0.0,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.007857,0,0
1446657,0.047140,4.574745e-04,0.377124,0.377124,0.0,0.000000,0.597112,0.000000,0.597112,0.0,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.007857,0,0
1446658,0.047140,2.151176e-05,0.377124,0.377124,0.0,0.000000,0.597112,0.000000,0.597112,0.0,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.007857,0,0
1446659,0.047140,2.610717e-04,0.377124,0.377124,0.0,0.000000,0.597112,0.000000,0.597112,0.0,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.007857,0,0
